# Minecraft for semantic segmentation

Fine-tuning SegFormer-B2 on Minecraft frames and measuring what transfers to
Cityscapes. The pipeline that produced the frames is in `pipeline/`; everything
here imports `segformer/`.

In [ ]:
!pip install -q transformers==4.44.2 "albumentations==1.4.14" "albucore==0.0.16"

from pathlib import Path

from torch.utils.data import DataLoader

from pipeline.blocks import CLASS_NAMES
from segformer.config import EvalConfig, TrainConfig, device, seed_all
from segformer.data import (CityscapesValDataset, MinecraftSegDataset,
                            cityscapes_images, cityscapes_transform,
                            split_pairs, train_transform, val_transform)
from segformer.evaluate import (evaluate, evaluate_baseline, evaluate_cityscapes,
                                print_results)
from segformer.plots import plot_confusion_matrix, plot_training_curves
from segformer.remap import ade20k_lut
from segformer.train import build_model, load_checkpoint, train

cfg, eval_cfg = TrainConfig(), EvalConfig()
seed_all(cfg.seed)
dev = device()
dev

In [ ]:
DATA = Path("data/dataset")            # holds rgb/ and mask_label/
CITYSCAPES = Path("data/cityscapes")   # holds leftImg8bit/ and gtFine/
CHECKPOINTS = Path("checkpoints")

## Data

300 paired frames from the Frankfurt-derived world, split 270/30.

In [ ]:
train_pairs, val_pairs = split_pairs(DATA / "rgb", DATA / "mask_label", cfg)

train_ds = MinecraftSegDataset(train_pairs, DATA / "rgb", DATA / "mask_label",
                               train_transform(cfg))
val_ds = MinecraftSegDataset(val_pairs, DATA / "rgb", DATA / "mask_label",
                             val_transform(cfg))
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=4, num_workers=2, pin_memory=True)

len(train_ds), len(val_ds)

## Training

SegFormer-B2 from the ADE20K checkpoint, with a fresh four-class head. The
backbone runs at 6e-6 and the head at 6e-5, decayed linearly over 40 epochs.
We keep the checkpoint with the best held-out mIoU.

In [ ]:
model = build_model().to(dev)
history = train(model, train_loader, val_loader, evaluate, cfg, dev, CHECKPOINTS)

In [ ]:
plot_training_curves(history)

## In-distribution check

The held-out 30 frames, to separate domain shift from a training failure.

In [ ]:
model = load_checkpoint(CHECKPOINTS / "best.pt", dev)
evaluate(model, val_loader, dev)

## Transfer to Cityscapes

The Cityscapes validation set, 500 images across Frankfurt, Lindau and Münster,
remapped to the four classes and evaluated at 1024x512.

In [ ]:
image_root = CITYSCAPES / "leftImg8bit" / "val"
label_root = CITYSCAPES / "gtFine" / "val"
city_ds = CityscapesValDataset(cityscapes_images(image_root), image_root,
                               label_root, cityscapes_transform(eval_cfg))
city_loader = DataLoader(city_ds, batch_size=eval_cfg.batch_size, num_workers=2,
                         pin_memory=True)

city_results = evaluate_cityscapes(model, city_loader, dev)
print_results("Minecraft -> Cityscapes", city_results)

In [ ]:
plot_confusion_matrix(city_results["hist"])

## ADE20K baseline

The same architecture from its public ADE20K weights, its 150-class output
mapped onto our four. Pixels where it predicts a class with no mapping are
dropped rather than counted wrong, so `coverage` reports how much survived.

In [ ]:
from transformers import SegformerForSemanticSegmentation

from segformer.config import MODEL_NAME

baseline = SegformerForSemanticSegmentation.from_pretrained(MODEL_NAME).to(dev).eval()
baseline_results = evaluate_baseline(baseline, city_loader, ade20k_lut(), dev)
print_results("ADE20K -> Cityscapes", baseline_results)

## Results

In [ ]:
print(f"{'class':<12}{'baseline':>10}{'ours':>10}{'delta':>10}")
for i, name in enumerate(CLASS_NAMES):
    b = baseline_results["per_class_iou"][i]
    o = city_results["per_class_iou"][i]
    print(f"{name:<12}{b:>10.4f}{o:>10.4f}{o - b:>+10.4f}")
print(f"{'mIoU':<12}{baseline_results['miou']:>10.4f}{city_results['miou']:>10.4f}"
      f"{city_results['miou'] - baseline_results['miou']:>+10.4f}")